[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlnjsh/rl-basics/blob/main/Section_9_deep_q_learning_complete.ipynb)

<div style="text-align:center">
    <h1>
        Deep Q-Learning
    </h1>
</div>

<br><br>

<div style="text-align:center">

In this notebook, we extend the Q-Learning algorithm to use function approximators (Neural Networks). The resulting algorithm is known as Deep Q-Learning.
</div>



<br>

In [ ]:
# === Setup: load the shared course modules =====================================
# The long environment-definition cell has been moved into two files that live
# next to this notebook: `envs.py` (the Maze environment) and `utils.py` (all the
# plotting / evaluation helpers). That keeps every lesson focused on the algorithm.
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('envs.py'):
    # On Google Colab there are no local files, so install the pinned gym version
    # and download the two helper modules from the course repository.
    !pip install -qq gym==0.23.0 pygame
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/envs.py
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/utils.py

from envs import Maze              # The 5x5 grid-world environment.
from utils import test_agent, plot_stats, seed_everything


## Import the necessary software libraries:

In [ ]:
# Standard Python tools used throughout the notebook.
import random                        # used to randomly sample transitions from the replay buffer
import copy                          # used to make an exact copy of the Q-network (the target network)
import gym                           # the OpenAI Gym library that provides the CartPole environment
import torch                         # PyTorch: builds the neural network and does the math on tensors
import torch.nn.functional as F      # functional helpers such as the mean-squared-error loss
import matplotlib.pyplot as plt      # for drawing the environment and result plots
from torch import nn as nn           # neural-network building blocks (layers, activations, containers)
from torch.optim import AdamW        # the optimiser that updates the network weights
from tqdm import tqdm                # draws a progress bar while training runs

## Create and prepare the environment

### Create the environment

In [ ]:
# Create the CartPole environment: a cart that must balance a pole by moving left/right.
env = gym.make('CartPole-v0')
# Seed all the random number generators so the run is reproducible.
seed_everything(env)
# Put the environment in its starting state.
env.reset()
# Render the current frame as an image and show it.
plt.imshow(env.render(mode='rgb_array'))

In [ ]:
# How many numbers describe a state (CartPole uses 4: cart position, cart velocity, pole angle, pole velocity).
state_dims = env.observation_space.shape[0]
# How many actions the agent can take (CartPole has 2: push left or push right).
num_actions = env.action_space.n
print(f"CartPole env: State dimensions: {state_dims}, Number of actions: {num_actions}")

### Prepare the environment to work with PyTorch

In [ ]:
# A thin wrapper that converts Gym's NumPy arrays into PyTorch tensors with the right shape,
# so the rest of the notebook can feed states straight into the neural network.
class PreprocessEnv(gym.Wrapper):

    def __init__(self, env):
        # Keep all the behaviour of the original environment.
        gym.Wrapper.__init__(self, env)

    def reset(self):
        # Reset the environment and get the raw starting observation (a NumPy array).
        obs = self.env.reset()
        # Turn it into a float tensor and add a batch dimension at the front (shape 1 x state_dims).
        return torch.from_numpy(obs).unsqueeze(dim=0).float()

    def step(self, action):
        # The action arrives as a tensor; pull out the plain Python integer Gym expects.
        action = action.item()
        # Take the action in the underlying environment.
        next_state, reward, done, info = self.env.step(action)
        # Convert the next state to a float tensor with a batch dimension.
        next_state = torch.from_numpy(next_state).unsqueeze(dim=0).float()
        # Wrap the reward as a 1x1 float tensor.
        reward = torch.tensor(reward).view(1, -1).float()
        # Wrap the done flag as a 1x1 tensor (True when the episode has ended).
        done = torch.tensor(done).view(1, -1)
        return next_state, reward, done, info

In [ ]:
# Wrap the environment so every state/reward/done it returns is already a PyTorch tensor.
env = PreprocessEnv(env)

In [ ]:
# Quick sanity check: reset, take one action, and print what we get back.
state = env.reset()
# Action 0 means 'push the cart left'.
action = torch.tensor(0)
# Step the environment once with that action.
next_state, reward, done, _ = env.step(action)
print(f"Sample state: {state}")
print(f"Next state: {next_state}, Reward: {reward}, Done: {done}")

## Create the Q-Network and policy

<br><br>

### The big idea, in plain words

Plain Q-Learning keeps a giant lookup table of "how good is each action in each state". CartPole's states are real numbers, so the table would be infinite. **Deep Q-Learning replaces the table with a neural network** that *guesses* those values from the state.

Two clever tricks keep the training stable:

1. **Experience replay** -- remember past moves and learn from a random mix of them, not just the latest move.
2. **A target network** -- a frozen copy of the network used to set the learning target, so we are not chasing a constantly moving goal.

The rest of the notebook builds exactly these two pieces and then the training loop that uses them.

### Create the Q-Network: $\hat q(s,a\,|\,\theta)$

Classic tabular Q-Learning stores one number for every (state, action) pair. That is impossible when states are continuous (CartPole has infinitely many). The fix in **Deep Q-Learning** is to *approximate* the Q-table with a small neural network with weights $\theta$.

The network takes a state in and outputs **one Q-value per action**. We then pick the action with the highest predicted value.

In [ ]:
# The Q-network: a small fully-connected network that maps a state to one Q-value per action.
# Q(s, a) estimates the total future reward of taking action a in state s.
q_network = nn.Sequential(
    nn.Linear(state_dims, 128),   # input layer: 4 state numbers -> 128 hidden units
    nn.ReLU(),                    # non-linear activation so the network can learn complex shapes
    nn.Linear(128, 64),           # hidden layer: 128 -> 64 units
    nn.ReLU(),                    # another ReLU activation
    nn.Linear(64, num_actions))   # output layer: 64 -> one Q-value for each action

### Create the target Q-Network: $\hat q(s, a\,|\,\theta_{targ})$

When we train, the target we chase depends on the same network we are updating. That moving-target problem makes training unstable.

The trick: keep a **second, frozen copy** of the network (the *target network*) to compute the training target. We only refresh its weights every few episodes. This gives a stable goal to aim at and is one of the two key ideas that made Deep Q-Learning work.

In [ ]:
# The target network is a frozen copy of the Q-network used to compute stable training targets.
# .eval() puts it in evaluation mode; we only refresh its weights occasionally (see below).
target_q_network = copy.deepcopy(q_network).eval()

### Create the exploratory policy: $b(s)$

In [ ]:
# Epsilon-greedy policy: mostly pick the best-known action, but explore at random sometimes.
def policy(state, epsilon=0.):
    # With probability epsilon, take a completely random action (exploration).
    if torch.rand(1) < epsilon:
        return torch.randint(num_actions, (1, 1))
    else:
        # Otherwise ask the Q-network for the value of each action (no gradient needed here).
        av = q_network(state).detach()
        # Pick the action with the highest estimated value (exploitation).
        return torch.argmax(av, dim=-1, keepdim=True)

## Create the Experience Replay buffer

The second key idea of DQN is **experience replay**. Instead of learning from each step the instant it happens, we store every transition $(s, a, r, done, s')$ in a big memory buffer and later train on **random mini-batches** drawn from it.

Why this helps:
- Consecutive steps in an episode are highly correlated; random sampling breaks that correlation, which neural networks dislike.
- Each experience can be reused many times, so we learn more from the same amount of play.

<br>
<div style="text-align:center">
    <p>A simple buffer that stores transitions of arbitrary values, adapted from
    <a href="https://pytorch.org/tutorials/intermediate/reinforcement_q_learning.html#training">this source.</a></p>
</div>


In [ ]:
# Experience Replay buffer: a fixed-size memory of past transitions we can sample from.
# Training on random past experiences breaks the correlation between consecutive steps.
class ReplayMemory:

    def __init__(self, capacity=100000):
        # Maximum number of transitions to keep.
        self.capacity = capacity
        # The list that actually stores the transitions.
        self.memory = []
        # Where the next transition will be written (acts like a circular pointer).
        self.position = 0

    def insert(self, transition):
        # Grow the list until it reaches capacity.
        if len(self.memory) < self.capacity:
            self.memory.append(None)
        # Store the transition at the current position.
        self.memory[self.position] = transition
        # Advance the pointer, wrapping back to 0 when the buffer is full (oldest gets overwritten).
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        # Only sample if we have enough experience stored.
        assert self.can_sample(batch_size)

        # Pick a random batch of transitions.
        batch = random.sample(self.memory, batch_size)
        # Regroup: turn a list of [s, a, r, d, s'] into separate lists of states, actions, etc.
        batch = zip(*batch)
        # Stack each group into one big tensor and return them.
        return [torch.cat(items) for items in batch]

    def can_sample(self, batch_size):
        # Wait until the buffer holds at least 10 batches before training (more varied samples).
        return len(self.memory) >= batch_size * 10

    def __len__(self):
        # Lets us call len(memory) to see how many transitions are stored.
        return len(self.memory)

## Implement the algorithm

We now put the pieces together into **Deep Q-Learning**. For each step we:

1. Pick an action with an $\epsilon$-greedy policy (explore a little, exploit a lot).
2. Store the transition in the replay buffer.
3. Sample a random batch and form the **TD target**
   $$y = r + \gamma \, \max_{a'} \hat q(s', a' \,|\, \theta_{targ})$$
   using the **frozen target network** for the $\max$ term.
4. Nudge the live network so its prediction $\hat q(s, a\,|\,\theta)$ moves toward $y$ (mean-squared-error loss).
5. Every few episodes, copy the live weights into the target network.

The `~done * gamma * next_qsa` term zeroes the future value when an episode has ended, because there is no "next state" to bootstrap from.

</br></br>

In [ ]:
# The full Deep Q-Learning training loop.
def deep_q_learning(q_network, policy, episodes,
                    alpha=0.0001, batch_size=32, gamma=0.99, epsilon=0.2):

    # AdamW updates the Q-network weights; alpha is the learning rate.
    optim = AdamW(q_network.parameters(), lr=alpha)
    # Create the experience replay buffer.
    memory = ReplayMemory()
    # Keep track of the loss and the return (total reward) of each episode for plotting.
    stats = {'MSE Loss': [], 'Returns': []}

    # Play many episodes; tqdm shows a progress bar.
    for episode in tqdm(range(1, episodes + 1)):
        # Start a fresh episode.
        state = env.reset()
        done = False
        ep_return = 0
        # Keep acting until the pole falls (the episode ends).
        while not done:
            # Choose an action with the epsilon-greedy policy.
            action = policy(state, epsilon)
            # Take the action and observe the result.
            next_state, reward, done, _ = env.step(action)

            # Store this transition in the replay buffer for later training.
            memory.insert([state, action, reward, done, next_state])

            # Once we have enough experience, do one gradient update.
            if memory.can_sample(batch_size):
                # Sample a random batch of past transitions.
                state_b, action_b, reward_b, done_b, next_state_b = memory.sample(batch_size)
                # Q-values the current network predicts for the actions we actually took.
                qsa_b = q_network(state_b).gather(1, action_b)

                # Ask the (frozen) target network for the next-state Q-values.
                next_qsa_b = target_q_network(next_state_b)
                # Take the best action's value in each next state: max over actions.
                next_qsa_b = torch.max(next_qsa_b, dim=-1, keepdim=True)[0]

                # The TD target: reward now, plus discounted best future value (zeroed if the episode ended).
                target_b = reward_b + ~done_b * gamma * next_qsa_b
                # Loss = how far our predictions are from the target.
                loss = F.mse_loss(qsa_b, target_b)
                # Clear old gradients.
                q_network.zero_grad()
                # Back-propagate the loss to get new gradients.
                loss.backward()
                # Update the weights one step.
                optim.step()

                # Record the loss for plotting.
                stats['MSE Loss'].append(loss.item())

            # Move to the next state.
            state = next_state
            # Add this step's reward to the episode total.
            ep_return += reward.item()


        # Save the episode's total reward.
        stats['Returns'].append(ep_return)

        # Every 10 episodes, copy the live network's weights into the target network.
        if episode % 10 == 0:
            target_q_network.load_state_dict(q_network.state_dict())

    return stats

In [ ]:
# Train the agent for 500 episodes and collect the statistics.
stats = deep_q_learning(q_network, policy, 500)

## Show results

### Plot execution stats

In [ ]:
# Plot the loss and the episode returns over training.
plot_stats(stats)

### Test the resulting agent

In [ ]:
# Watch the trained agent play 2 episodes greedily (no exploration).
test_agent(env, policy, episodes=2)

## Resources

[[1] Playing Atari with Deep Reinforcement Learning](https://www.cs.toronto.edu/~vmnih/docs/dqn.pdf)